<a href="https://colab.research.google.com/github/Lordyblade/BigData26_B_2411532010_IbrahimMousaDhani/blob/main/Praktikum4/BD_P04_2411532010_IbrahimMousaDhani.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Praktikum 4 — NoSQL dan Pemodelan Data**

Praktikum ini memperkenalkan cara lain menyimpan data. Kita mengambil satu data yang sama yaitu perjalanan
taksi, lalu menyimpannya dengan beberapa cara berbeda, mengukur sendiri mana yang cepat untuk
pertanyaan apa, dan mana yang justru lambat.

### **K-1. Menyiapkan Lingkungan dan Data Contoh**
Memasang pustaka, membaca trips_bersih/, dan mengganti nama kolom menjadi nama yang
mudah dibaca. Sel pertama khusus Colab; sel kedua berlaku di mana saja.

In [ ]:
!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"

from google.colab import drive
drive.mount("/content/drive")

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"
TRIPS_BERSIH = f"{DIR_P3}/trips_bersih"    # hasil Praktikum 3 (K-10)

import os
os.makedirs(DIR_SIMPAN, exist_ok=True)
assert os.path.exists(TRIPS_BERSIH), "Folder trips_bersih/ tidak ada. Jalankan ulang P3/K-10 atau pakai K-1b."
import os, time, json, sqlite3, random, shutil, glob
import numpy as np, pandas as pd

# Pengukur waktu (dari Praktikum 1/K-3, versi ringkas)
catatan = []
def ukur(label, fungsi):
    t0 = time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 4)})
    print(f"[{label}] {detik:.4f} s")
    return hasil

def baca_partisi_manual():
    semua_df = []
    # Membaca file parquet satu per satu dari sub-folder
    for file in glob.glob(f"{TRIPS_BERSIH}/**/*.parquet", recursive=True):
        df_temp = pd.read_parquet(file)

        # Ekstrak nama borough dari nama folder (misal: dari "borough_naik=Staten%20Island" jadi "Staten Island")
        nama_folder = os.path.basename(os.path.dirname(file))
        if "=" in nama_folder:
            wilayah = nama_folder.split("=")[1].replace("%20", " ")
            df_temp["borough_naik"] = wilayah

        semua_df.append(df_temp)

    # Gabungkan semua data menjadi satu DataFrame utuh
    return pd.concat(semua_df, ignore_index=True)

# Panggil fungsi pengukur waktu dengan fungsi baca manual kita
df = ukur("baca trips_bersih", baca_partisi_manual)

# Ganti nama kolom asli data TLC menjadi nama yang lebih mudah dibaca
NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}
df = df.rename(columns=NAMA_BARU)
df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)
print("baris:", f"{len(df):,}", "| kolom:", df.shape[1])

# Contoh 300.000 baris untuk K-2 dan K-3 (agar tidak terlalu lama)
sample = df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index + 1
sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 19.5 MB/s eta 0:00:00
Mounted at /content/drive
[baca trips_bersih] 23.1347 s
baris: 2,986,910 | kolom: 18


### **K-2. Tabel Biasa vs Key-value**
Menyimpan perjalanan yang sama dalam dua cara, lalu mengajukan dua pertanyaan berbeda.
Cara pertama: tabel SQLite. Cara kedua: key-value, yaitu kamus Python yang kuncinya “perjalanan:<id>”
dan nilainya teks JSON.

In [ ]:
KOLOM = ["id_perjalanan", "waktu_jemput", "zona_jemput", "zona_tujuan",
         "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data = sample[KOLOM].copy()

# Model 1 - relasional: tabel SQLite
if os.path.exists("uji.db"):
    os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur("relasional: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))

# Model 2 - key-value: kamus (dict) dengan kunci "perjalanan:<id>", nilainya teks JSON
def bangun_kv():
    return {f"perjalanan:{r['id_perjalanan']}": json.dumps(r) for r in data.to_dict("records")}

kv = ukur("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))
# Pertanyaan A: ambil SATU perjalanan berdasarkan id (200 id acak)
random.seed(42)
id_uji = random.sample(range(1, len(data) + 1), 200)

def cari_sql():
    cur = con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for
i in id_uji]

def cari_kv():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji]

r1 = ukur("cari 200 perjalanan - SQL tanpa indeks", cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan(id_perjalanan)")    # buat indeks
r2 = ukur("cari 200 perjalanan - SQL dengan indeks", cari_sql)
r3 = ukur("cari 200 perjalanan - key-value", cari_kv)
print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])

# Pertanyaan B: hitung rata-rata total bayar per wilayah (melibatkan SEMUA data)
def hitung_sql():
    return con.execute("""SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
                          FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC,
1""").fetchall()

def hitung_kv():
    total, jumlah = {}, {}
    for teks in kv.values():
        d = json.loads(teks)              # setiap nilai harus dibuka dulu
        w = d["wilayah_jemput"]
        total[w] = total.get(w, 0) + d["total_bayar"]
        jumlah[w] = jumlah.get(w, 0) + 1
    return sorted([(w, jumlah[w], round(total[w] / jumlah[w], 2)) for w in jumlah],
                  key=lambda x: (-x[1], x[0]))

h1 = ukur("hitung per wilayah - SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah - key-value (buka semua nilai)", hitung_kv)
print(h1[:3])
print("hasil sama?", h1 == h2)


[relasional: muat tabel] 1.1051 s
[key-value: bangun dict] 3.6001 s
jumlah key: 300000
[cari 200 perjalanan - SQL tanpa indeks] 12.5739 s
[cari 200 perjalanan - SQL dengan indeks] 0.0035 s
[cari 200 perjalanan - key-value] 0.0011 s
hasil sama? True
[hitung per wilayah - SQL GROUP BY] 0.2148 s
[hitung per wilayah - key-value (buka semua nilai)] 1.3041 s
[('Manhattan', 266988, 22.74), ('Queens', 27266, 70.28), ('Unknown', 3825, 40.06)]
hasil sama? True


### **K-3. Document: Bentuk Data yang Fleksibel**
Menyimpan perjalanan sebagai dokumen JSON bersarang, lalu melihat dua hal: data terkait
dibungkus dalam satu dokumen (embedding), dan field tip sengaja hanya ada pada pembayaran kartu,
mirip data TLC asli yang tidak mencatat tip tunai.

In [ ]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
    dok = {
        "id_perjalanan": int(r.id_perjalanan),
        "waktu_jemput": r.waktu_jemput,
        "jemput": {"zona": int(r.zona_jemput), "wilayah": r.wilayah_jemput},
        "tujuan": {"zona": int(r.zona_tujuan)},
        "biaya":  {"jarak_mil": float(r.jarak_mil),
                   "total": float(r.total_bayar),
                   "metode_bayar": r.metode_bayar},
    }
    # Tip hanya dicatat untuk pembayaran kartu -> field ini ada di sebagian dokumen saja
    if r.metode_bayar == "Kartu kredit":
        dok["biaya"]["tip"] = float(r.tip)
    return dok

db = TinyDB(storage=MemoryStorage)
tabel = db.table("perjalanan")
docs = [ke_dokumen(r) for r in sample.head(20_000).itertuples()]
tabel.insert_multiple(docs)
print("jumlah dokumen:", len(tabel))
print(json.dumps(tabel.all()[0], indent=2))

jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-15 18:44:37",
  "jemput": {
    "zona": 170,
    "wilayah": "Manhattan"
  },
  "tujuan": {
    "zona": 100
  },
  "biaya": {
    "jarak_mil": 0.6,
    "total": 11.55,
    "metode_bayar": "Kartu kredit",
    "tip": 1.05
  }
}


In [ ]:
Q = Query()
print("jemput di Manhattan & total > 50 :", len(tabel.search((Q.jemput.wilayah == "Manhattan") &
(Q.biaya.total > 50))))
print("dokumen yang punya field tip     :", len(tabel.search(Q.biaya.tip.exists())))
print("dokumen yang TIDAK punya tip     :", len(tabel.search(~Q.biaya.tip.exists())))

# Menambah field baru ("cuaca") pada satu dokumen - tanpa mengubah struktur apa pun
tabel.insert({"id_perjalanan": 999_999, "waktu_jemput": "2023-01-15 08:00:00",
              "jemput": {"zona": 161, "wilayah": "Manhattan"}, "tujuan": {"zona": 236},
              "biaya": {"jarak_mil": 1.2, "total": 14.5, "metode_bayar": "Kartu kredit", "tip":
3.0},
              "cuaca": {"hujan": True, "suhu_c": -2.0}})
print("dokumen yang punya field cuaca   :", len(tabel.search(Q.cuaca.exists())))
print("total dokumen sekarang           :", len(tabel))

jemput di Manhattan & total > 50 : 661
dokumen yang punya field tip     : 15910
dokumen yang TIDAK punya tip     : 4090
dokumen yang punya field cuaca   : 1
total dokumen sekarang           : 20001


### **K-4. Penyimpanan per Kolom: DuckDB pada Parquet**
Menjalankan SQL langsung pada folder Parquet berpartisi tanpa memuatnya ke tabel dulu, lalu
membandingkannya dengan SQLite (penyimpanan per baris) untuk perhitungan yang sama. View pada
kode ini adalah “jendela” bernama di atas file Parquet; data tidak disalin.

In [ ]:
import duckdb

GLOB = f"{TRIPS_BERSIH}/*/*.parquet"
con_d = duckdb.connect()

# VIEW = "jendela" bernama yang membaca Parquet langsung (tidak menyalin data).
# Di sinilah nama kolom asli TLC diberi nama yang mudah dibaca.
con_d.sql(f"""
    CREATE VIEW perjalanan AS
    SELECT tpep_pickup_datetime AS waktu_jemput,
           PULocationID         AS zona_jemput,
           DOLocationID         AS zona_tujuan,
           borough_naik         AS wilayah_jemput,
           trip_distance        AS jarak_mil,
           total_amount         AS total_bayar,
           nama_pembayaran      AS metode_bayar
    FROM read_parquet('{GLOB}', hive_partitioning=true)""")

SQL = """SELECT wilayah_jemput, COUNT(*) AS jumlah, ROUND(AVG(total_bayar), 2) AS rata_bayar
         FROM perjalanan GROUP BY wilayah_jemput ORDER BY jumlah DESC, wilayah_jemput"""
hasil_d = ukur("DuckDB - hitung per wilayah (langsung dari Parquet)", lambda:
con_d.sql(SQL).df())
print(hasil_d)

# Pembanding: SQLite (menyimpan per baris) pada data yang sama
KOLOM_S = ["wilayah_jemput", "total_bayar", "jarak_mil", "zona_jemput", "zona_tujuan"]
if os.path.exists("baris.db"):
    os.remove("baris.db")
con_s = sqlite3.connect("baris.db")
ukur("SQLite - muat seluruh baris ke tabel", lambda: df[KOLOM_S].to_sql("perjalanan", con_s,
index=False))
hasil_s = ukur("SQLite - hitung per wilayah", lambda: con_s.execute(
    """SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
       FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall())
print("hasil sama?", [tuple(r) for r in hasil_d.itertuples(index=False)] == [tuple(r) for r in
hasil_s])

[DuckDB - hitung per wilayah (langsung dari Parquet)] 0.3748 s
               wilayah_jemput   jumlah  rata_bayar
0                   Manhattan  2659609       22.72
1                      Queens   270315       70.46
2                     Unknown    37609       39.24
3                    Brooklyn    15724       32.72
4                       Bronx     3074       34.44
5  __HIVE_DEFAULT_PARTITION__      340       74.72
6               Staten Island      211       74.07
7                         EWR       28      106.41
[SQLite - muat seluruh baris ke tabel] 14.4181 s
[SQLite - hitung per wilayah] 2.9043 s
hasil sama? True


In [ ]:
# Filter pada kolom partisi: DuckDB hanya membuka folder wilayah yang diminta (partition pruning, P3)
SQL2 = """SELECT COUNT(*), ROUND(AVG(total_bayar), 2)
          FROM perjalanan WHERE wilayah_jemput = 'Staten Island'"""
print(ukur("DuckDB - hanya wilayah Staten Island", lambda: con_d.sql(SQL2).fetchall()))
ukuran_pq = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(TRIPS_BERSIH) for f
in fs) / 1024**2
print(f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | SQLite ({len(KOLOM_S)} kolom): {os.path.getsize('baris.db') / 1024**2:.1f} MB")

[DuckDB - hanya wilayah Staten Island] 0.0498 s
[(211, 74.07)]
Parquet (18 kolom): 66.5 MB | SQLite (5 kolom): 114.2 MB


### **K-5. Graph: Pertanyaan tentang Keterhubungan**
Meringkas perjalanan menjadi arus antar zona. Setiap zona adalah node, dan setiap pasangan
zona asal-tujuan adalah edge berarah dengan bobot jumlah perjalanan. Hasil graph disilangkan dengan
SQL agar kita yakin keduanya menjawab hal yang sama.

In [ ]:
import networkx as nx

# Satu baris = satu arus: berapa perjalanan dari zona jemput ke zona tujuan tertentu
arus = ukur("hitung arus antar zona", lambda: con_d.sql("""
    SELECT zona_jemput AS asal, zona_tujuan AS tujuan, COUNT(*) AS jumlah
    FROM perjalanan GROUP BY 1, 2""").df())

# Zona = node (titik), arus = edge (garis berarah, bobotnya jumlah perjalanan)
G = nx.DiGraph()
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))
print("node (zona):", G.number_of_nodes(), "| edge (arus):", G.number_of_edges())

# Lima arus terbesar. Urutan dibuat pasti dengan pengurut tambahan (jumlah, asal, tujuan)
top_graf = sorted(G.edges(data=True), key=lambda e: (-e[2]["jumlah"], e[0], e[1]))[:5]
print("graph:", [(a, b, d["jumlah"]) for a, b, d in top_graf])
top_sql = arus.sort_values(["jumlah", "asal", "tujuan"], ascending=[False, True, True]).head(5)
print("SQL  :", list(top_sql.itertuples(index=False, name=None)))

# Lima zona yang paling banyak menjadi tujuan
populer = sorted(G.in_degree(weight="jumlah"), key=lambda kv: (-kv[1], kv[0]))[:5]
print("zona tujuan terpopuler:", populer)

[hitung arus antar zona] 0.5541 s
node (zona): 262 | edge (arus): 21583
graph: [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
SQL  : [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
zona tujuan terpopuler: [(236, 144209), (237, 130369), (161, 113991), (230, 87706), (170, 87110)]


In [ ]:
MIN = 50                       # abaikan arus yang terlalu jarang (kurang dari 50 perjalanan)
zona_awal = populer[0][0]      # mulai dari zona tujuan terpopuler

# Pertanyaan: zona mana yang bisa dicapai dalam TEPAT 2 langkah dari zona_awal?
# (1) cara graph: telusuri tetangga dari tetangga
def dua_langkah_graph():
    hasil = set()
    for b, d1 in G[zona_awal].items():
        if d1["jumlah"] < MIN:
            continue
        for c, d2 in G[b].items():
            if d2["jumlah"] >= MIN and c != zona_awal:
                hasil.add(c)
    return hasil

# (2) cara SQL: harus menggabungkan tabel arus dengan dirinya sendiri (self-join)
con_d.register("arus", arus)
def dua_langkah_sql():
    return set(con_d.sql(f"""
        SELECT DISTINCT a2.tujuan
        FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
        WHERE a1.asal = {zona_awal} AND a1.jumlah >= {MIN}
          AND a2.jumlah >= {MIN} AND a2.tujuan <> {zona_awal}""").df()["tujuan"])

hg = ukur("2 langkah - graph (NetworkX)", dua_langkah_graph)
hs = ukur("2 langkah - SQL self-join (DuckDB)", dua_langkah_sql)
print("zona awal:", zona_awal, "| terjangkau (graph):", len(hg), "| (SQL):", len(hs), "| sama?",
hg == hs)

# (3) Jumlah langkah bebas: "dalam paling banyak k langkah" - cukup ganti angka k
H = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])
for k in (1, 2, 3, 4):
    n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
    print(f"dalam <= {k} langkah: {n} zona")

[2 langkah - graph (NetworkX)] 0.0087 s
[2 langkah - SQL self-join (DuckDB)] 0.0177 s
zona awal: 236 | terjangkau (graph): 214 | (SQL): 214 | sama? True
dalam <= 1 langkah: 74 zona
dalam <= 2 langkah: 214 zona
dalam <= 3 langkah: 214 zona
dalam <= 4 langkah: 214 zona


### **K-6. Indeks Spasial**
Mencari titik jemput di dalam 1.000 kotak kecil (sekitar 1 km × 1 km) dengan tiga cara: loop
Python, NumPy tanpa indeks spasial, dan indeks STRtree. Cara pertama hanya dijalankan untuk 5 kotak
karena terlalu lambat.

In [ ]:
from shapely import STRtree, box, points

# CATATAN: data TLC 2023 hanya berisi ID zona, TANPA koordinat. Titik di bawah ini SINTETIS.
rng = np.random.default_rng(42)
N = 200_000
bujur = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70)     # longitude
lintang = rng.normal(40.74, 0.05, N).clip(40.50, 40.92)      # latitude
titik = points(bujur, lintang)

# 1.000 wilayah pencarian berupa kotak kecil (sekitar 1 km x 1 km)
M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)
kotak = [box(x - 0.005, y - 0.005, x + 0.005, y + 0.005) for x, y in zip(cx, cy)]

# Cara 1: periksa titik satu per satu dengan loop Python (hanya 5 kotak, 1.000 akan sangat lama)
def cara_loop():
    return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]

# Cara 2: bandingkan seluruh titik sekaligus dengan NumPy (tanpa indeks spasial), 1.000 kotak
def cara_numpy():
    hasil = []
    for x, y in zip(cx, cy):
        m = (bujur >= x - 0.005) & (bujur <= x + 0.005) & (lintang >= y - 0.005) & (lintang <= y
+ 0.005)
        hasil.append(int(m.sum()))
    return hasil

h_loop = ukur("cara 1 - loop Python, 5 kotak", cara_loop)
h_np = ukur("cara 2 - NumPy, 1000 kotak", cara_numpy)

# Cara 3: indeks spasial STRtree (sejenis R-tree)
pohon = ukur("bangun indeks STRtree", lambda: STRtree(titik))
h_idx = ukur("cara 3 - STRtree, 1000 kotak",
             lambda: [len(pohon.query(k, predicate="contains")) for k in kotak])

print("5 kotak pertama - loop:", h_loop, "| STRtree:", h_idx[:5])
print("hasil NumPy dan STRtree sama?", h_np == h_idx, "| loop dan STRtree sama (5 pertama)?",
h_loop == h_idx[:5])
print("total titik ditemukan (1000 kotak):", sum(h_idx))


[cara 1 - loop Python, 5 kotak] 11.7339 s
[cara 2 - NumPy, 1000 kotak] 0.6332 s
[bangun indeks STRtree] 0.0745 s
[cara 3 - STRtree, 1000 kotak] 0.2492 s
5 kotak pertama - loop: [1012, 84, 228, 748, 509] | STRtree: [1012, 84, 228, 748, 509]
hasil NumPy dan STRtree sama? True | loop dan STRtree sama (5 pertama)? True
total titik ditemukan (1000 kotak): 417545


### **K-7. Delta Lake: Versi, Time Travel, dan Schema Enforcement**
Menjawab tiga pertanyaan dari Praktikum 3/K-8. Tabel Delta dibuat, ditambah data, lalu
dikoreksi dengan update; setelah itu kita menelusuri versinya.

In [ ]:
from deltalake import DeltaTable, write_deltalake
import shutil
import os

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data_delta = df[KOLOM_DELTA].sample(n=min(200_000, len(df)), random_state=42).reset_index(drop=True)

# [PERUBAHAN 1]: Buat variabel untuk jalur lokal dan jalur Google Drive
PATH_DELTA_LOKAL = "/content/trips_delta" # <-- Delta Lake akan bekerja di sini dulu
PATH_DELTA_DRIVE = f"{DIR_SIMPAN}/trips_delta" # <-- Tujuan akhirnya di Drive

# Bersihkan folder lokal jika sudah ada sisa sebelumnya
if os.path.exists(PATH_DELTA_LOKAL):
    shutil.rmtree(PATH_DELTA_LOKAL)
# Bersihkan folder tujuan di Drive agar tidak bentrok saat dicopy
if os.path.exists(PATH_DELTA_DRIVE):
    shutil.rmtree(PATH_DELTA_DRIVE)

# [PERUBAHAN 2]: Ubah semua 'PATH_DELTA' menjadi 'PATH_DELTA_LOKAL'
# Versi 0: menulis data awal (100.000 baris)
write_deltalake(PATH_DELTA_LOKAL, data_delta.iloc[:100_000], mode="overwrite")
dt = DeltaTable(PATH_DELTA_LOKAL)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 1: menambah 50.000 baris
write_deltalake(PATH_DELTA_LOKAL, data_delta.iloc[100_000:150_000], mode="append")
dt = DeltaTable(PATH_DELTA_LOKAL)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 2: koreksi data. Anggap tarif Tunai di wilayah Bronx perlu ditambah 1.0
sekarang = data_delta.iloc[:150_000]
n_koreksi = int(((sekarang["wilayah_jemput"] == "Bronx") & (sekarang["metode_bayar"] == "Tunai")).sum())
print("baris yang akan dikoreksi:", n_koreksi)
dt.update(predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
          updates={"total_bayar": "total_bayar + 1.0"})
print("versi:", DeltaTable(PATH_DELTA_LOKAL).version())

# [PERUBAHAN 3]: Pindahkan hasil akhirnya ke Google Drive
print(f"\nMenyalin data Delta ke Google Drive ({PATH_DELTA_DRIVE})...")
shutil.copytree(PATH_DELTA_LOKAL, PATH_DELTA_DRIVE)
print("Selesai! Data Delta Lake berhasil diproses dan disimpan permanen.")

versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 26
versi: 2

Menyalin data Delta ke Google Drive (/content/drive/MyDrive/BigData/Praktikum4/trips_delta)...
Selesai! Data Delta Lake berhasil diproses dan disimpan permanen.


In [ ]:
PATH_DELTA = PATH_DELTA_LOKAL

# TIME TRAVEL: baca tabel sebagaimana keadaannya pada versi tertentu
v0 = DeltaTable(PATH_DELTA, version=0).to_pandas()
v1 = DeltaTable(PATH_DELTA, version=1).to_pandas()
terbaru = DeltaTable(PATH_DELTA).to_pandas()
print("baris v0:", len(v0), "| v1:", len(v1), "| terbaru:", len(terbaru))
selisih = terbaru["total_bayar"].sum() - v1["total_bayar"].sum()
print("selisih total_bayar (terbaru - v1):", round(selisih, 2), "| jumlah baris dikoreksi:",
n_koreksi)

# Riwayat perubahan tabel
for h in DeltaTable(PATH_DELTA).history():
    print(h.get("version"), h.get("operation"))

# Log transaksi: satu file JSON per versi
print(sorted(os.listdir(f"{PATH_DELTA}/_delta_log")))

# PENEGAKAN SKEMA: menambah data dengan kolom asing harus DITOLAK, tabel tidak berubah
tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1
try:
    write_deltalake(PATH_DELTA, tambahan, mode="append")
    print("append kolom asing: BERHASIL (tidak diharapkan)")
except Exception as e:
    print("append kolom asing DITOLAK:", type(e).__name__)
print("versi setelah percobaan gagal:", DeltaTable(PATH_DELTA).version())

baris v0: 100000 | v1: 150000 | terbaru: 150000
selisih total_bayar (terbaru - v1): 26.0 | jumlah baris dikoreksi: 26
2 UPDATE
1 WRITE
0 WRITE
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']
append kolom asing DITOLAK: SchemaMismatchError
versi setelah percobaan gagal: 2


### **K-8. Small File Problem pada Tabel Delta**
Meniru data yang masuk sedikit demi sedikit (menghasilkan banyak file kecil, seperti
Praktikum 3/K-6), lalu merapikannya dengan compaction.

In [ ]:
dt = DeltaTable(PATH_DELTA_LOKAL)
print("file data aktif sebelum:", len(dt.file_uris()), "| versi:", dt.version())

# 20 kali menambah data kecil (500 baris) - seperti data yang masuk sedikit demi sedikit
for i in range(20):
    potongan = data_delta.iloc[150_000 + i * 500: 150_000 + (i + 1) * 500]
    write_deltalake(PATH_DELTA_LOKAL, potongan, mode="append")
dt = DeltaTable(PATH_DELTA_LOKAL)
print("file data aktif setelah 20 append kecil:", len(dt.file_uris()), "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))

t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA_LOKAL).to_pandas(); t_sebelum = time.perf_counter() - t0

# COMPACTION: gabungkan file kecil menjadi file yang lebih besar (dicatat sebagai versi baru)
hasil = dt.optimize.compact()
print("hasil compact:", {k: hasil[k] for k in ("numFilesAdded", "numFilesRemoved")})

dt = DeltaTable(PATH_DELTA_LOKAL)
t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA_LOKAL).to_pandas(); t_sesudah = time.perf_counter() - t0
print("file data aktif setelah compact:", len(dt.file_uris()), "| versi:", dt.version(), "| baris:", len(dt.to_pandas()))
print(f"waktu baca sebelum compact: {t_sebelum:.4f} s | sesudah: {t_sesudah:.4f} s")

# File lama TIDAK langsung terhapus (masih dibutuhkan untuk time travel). VACUUM yang menghapusnya.
print("file parquet di disk:", len(glob.glob(f"{PATH_DELTA_LOKAL}/*.parquet")))
kandidat = dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=True)   # dry_run = hanya melaporkan
print("file yang akan dihapus VACUUM (dry run):", len(kandidat))

file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1 | versi: 23 | baris: 160000
waktu baca sebelum compact: 0.0670 s | sesudah: 0.0493 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


### **K-9. Merangkum dan Memilih**
Menyimpan semua angka waktu ke benchmark_nosql.csv

In [ ]:
rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print(rekap.to_string(index=False))

                                            langkah   detik
                                  baca trips_bersih 23.1347
                             relasional: muat tabel  1.1051
                             key-value: bangun dict  3.6001
             cari 200 perjalanan - SQL tanpa indeks 12.5739
            cari 200 perjalanan - SQL dengan indeks  0.0035
                    cari 200 perjalanan - key-value  0.0011
                  hitung per wilayah - SQL GROUP BY  0.2148
  hitung per wilayah - key-value (buka semua nilai)  1.3041
DuckDB - hitung per wilayah (langsung dari Parquet)  0.3748
               SQLite - muat seluruh baris ke tabel 14.4181
                        SQLite - hitung per wilayah  2.9043
               DuckDB - hanya wilayah Staten Island  0.0498
                             hitung arus antar zona  0.5541
                       2 langkah - graph (NetworkX)  0.0087
                 2 langkah - SQL self-join (DuckDB)  0.0177
                      cara 1 - loop Pyth

## **Analisis Hasil**

In [ ]:
r = dict(zip(rekap.langkah, rekap.detik))
print("1a. tanpa indeks / dengan indeks :", round(r["cari 200 perjalanan - SQL tanpa indeks"] / r["cari 200 perjalanan - SQL dengan indeks"], 1))
print("1b. dengan indeks / key-value    :", round(r["cari 200 perjalanan - SQL dengan indeks"] / r["cari 200 perjalanan - key-value"], 2))
print("2.  key-value / SQL GROUP BY     :", round(r["hitung per wilayah - key-value (buka semua nilai)"] / r["hitung per wilayah - SQL GROUP BY"], 2))
print("4.  SQLite / DuckDB              :", round(r["SQLite - hitung per wilayah"] / r["DuckDB - hitung per wilayah (langsung dari Parquet)"], 2))
loop_1000 = r["cara 1 - loop Python, 5 kotak"] / 5 * 1000
idx_total = r["bangun indeks STRtree"] + r["cara 3 - STRtree, 1000 kotak"]
print("6a. loop (perkiraan 1000 kotak, detik):", round(loop_1000, 1))
print("6b. loop / STRtree (total)       :", round(loop_1000 / idx_total, 1))
print("6c. NumPy / STRtree (total)      :", round(r["cara 2 - NumPy, 1000 kotak"] / idx_total, 2))
print("6d. NumPy / STRtree (tanpa bangun):", round(r["cara 2 - NumPy, 1000 kotak"] / r["cara 3 - STRtree, 1000 kotak"], 2))

1a. tanpa indeks / dengan indeks : 3592.5
1b. dengan indeks / key-value    : 3.18
2.  key-value / SQL GROUP BY     : 6.07
4.  SQLite / DuckDB              : 7.75
6a. loop (perkiraan 1000 kotak, detik): 2346.8
6b. loop / STRtree (total)       : 7249.9
6c. NumPy / STRtree (total)      : 1.96
6d. NumPy / STRtree (tanpa bangun): 2.54


**1. Pencarian satu data**

Tabel tanpa indeks ternyata 3.592,5 kali lebih lambat daripada tabel berindeks (12,5739 detik berbanding 0,0035 detik untuk 200 pencarian). Penyebabnya, tanpa indeks database tidak tahu letak sebuah id sehingga harus membuka baris satu per satu, sedangkan indeks sudah mencatat letak tiap nilai (M.1).

Tabel berindeks sendiri masih 3,18 kali lebih lambat daripada key-value (0,0035 detik berbanding 0,0011 detik). Hal ini wajar karena dict pada key-value sudah berfungsi sebagai indeks, jadi kunci langsung menunjukkan letak datanya.

Selisih 12,57 detik (3.592,5 kali) jelas berarti karena akan terasa oleh pengguna. Sebaliknya, selisih 3,18 kali tidak berarti pada satu komputer, sebab pada Latihan 1 perbedaannya hanya sekitar 12 mikrodetik per pencarian (17,83 berbanding 5,79 mikrodetik).

**2. Harga kesederhanaan**

Untuk menghitung rata-rata, key-value 6,07 kali lebih lambat daripada SQL GROUP BY (1,3041 detik berbanding 0,2148 detik pada 300.000 baris). Alasannya, bagi key-value setiap nilai hanyalah teks, sehingga semuanya harus dibuka dulu dengan `json.loads`. Pada SQL, database sudah tahu bahwa `total_bayar` adalah kolom angka dan bisa langsung menghitungnya (M.2). Inilah harga dari kesederhanaan key-value, yaitu hilangnya kemampuan mencari dan menghitung berdasarkan isi.

**3. Data fleksibel**

Keuntungannya, dokumen ke-20.001 bisa membawa field baru `cuaca` (hanya 1 dokumen yang memilikinya) tanpa ALTER TABLE atau migrasi apa pun. Hal ini terjadi karena document store tidak memaksakan skema di awal, jadi bentuk dokumen ditentukan oleh aplikasi.

Risikonya, kalau ada salah ketik nama field, misalnya "biya" sebagai ganti "biaya", database tidak akan menegur karena memang tidak ada skema yang dilanggar. Apalagi 4.090 dari 20.000 dokumen memang tidak punya field `tip`, sehingga dokumen yang kehilangan field akibat salah ketik terlihat sama saja dengan yang memang tidak memilikinya. Akibatnya kesalahan baru ketahuan belakangan, dan validasi menjadi tugas aplikasi.

**4. Penyimpanan per kolom**

DuckDB 7,75 kali lebih cepat daripada SQLite untuk perhitungan per wilayah (0,3748 detik berbanding 2,9043 detik). Ada dua penyebabnya.

Penyebab pertama berkaitan dengan format kolom. Query hanya perlu membaca kolom `total_bayar` dari 18 kolom (kolom `wilayah_jemput` diambil dari nama folder partisi), sedangkan SQLite menyimpan per baris sehingga bagian lain dari baris ikut terbaca (column pruning).

Penyebab kedua tidak berkaitan dengan format kolom. DuckDB memproses banyak baris sekaligus dan memakai beberapa inti prosesor. Selain itu, DuckDB tidak memerlukan tahap muat ke tabel, yang pada SQLite memakan 14,4181 detik.

**5. Graph**

Pada pencarian dua langkah, graph (0,0087 detik) dan SQL self-join (0,0177 detik) sama-sama selesai dalam hitungan milidetik pada 262 zona, jadi kecepatan bukan alasan utama memilih graph. Graph tetap layak karena cara bertanyanya lebih ringkas. Jumlah zona yang terjangkau untuk k = 1, 2, 3, dan 4 adalah 74, 214, 214, dan 214, dan semuanya didapat hanya dengan mengganti angka `cutoff=k`. Pada SQL, setiap langkah tambahan membutuhkan satu JOIN lagi (tiga langkah berarti dua self-join), sehingga query harus ditulis ulang. Hasil k = 1 sampai 4 juga memperlihatkan bahwa jangkauan sudah jenuh di 214 zona sejak langkah kedua.

**6. Indeks spasial**

STRtree 7.249,9 kali lebih cepat daripada loop Python. Perkiraan waktu loop untuk 1.000 kotak adalah 2.346,8 detik (11,7339 detik untuk 5 kotak, dikali 200, dan tidak dijalankan), sedangkan STRtree hanya butuh 0,3237 detik untuk membangun indeks dan mencari. Penyebabnya, loop memeriksa 200.000 titik satu per satu pada setiap kotak.

Terhadap NumPy, STRtree hanya 1,96 kali lebih cepat (0,6332 detik), atau 2,54 kali kalau waktu membangun indeks tidak dihitung. Hal ini karena NumPy sudah memeriksa semua titik sekaligus dengan operasi vektor, sehingga ruang percepatan yang tersisa kecil.

Karena itu, klaim percepatan harus selalu menyebut pembandingnya dan apakah waktu membangun indeks ikut dihitung. Perlu diingat juga bahwa hasil ini berasal dari titik tiruan.

**7. Delta Lake**

(a) Update tidak menghilangkan data lama. Selisih `total_bayar` antara versi terbaru dan versi 1 adalah 26,0, sama dengan jumlah baris yang dikoreksi (26), dan versi 1 tetap terbaca utuh dengan 150.000 baris. Hal ini terjadi karena update menulis file baru lalu mencatat bahwa file lama keluar dari tabel, tanpa menghapusnya (M.6).

(b) Penulisan yang gagal tidak meninggalkan jejak. Penambahan data dengan kolom asing ditolak dengan `SchemaMismatchError`, dan versi setelah percobaan gagal tetap 2. Perubahan baru dianggap sah setelah satu berkas komit selesai ditulis, sehingga penulisan yang ditolak tidak menghasilkan versi baru.

Setelah compaction, file di disk (24) masih lebih banyak daripada file aktif (1). Alasannya, file lama masih dibutuhkan agar versi sebelumnya tetap bisa dibaca lewat time travel, dan hanya VACUUM yang menghapusnya secara fisik. Hasil dry run menunjukkan ada 23 file kandidat yang bisa dihapus.

## **Studi Kasus**

Berikut rancangan penyimpanan untuk kelima fitur aplikasi taksi. Angka pada kolom alasan diambil dari `benchmark_nosql.csv` hasil ukur sendiri.

| Fitur | Cara menyimpan | Alasan | Kelemahan yang perlu diwaspadai |
|---|---|---|---|
| 1. Detail perjalanan berdasarkan id | Key-value, atau tabel dengan indeks pada id | Untuk 200 pencarian, key-value butuh 0,0011 detik dan tabel berindeks 0,0035 detik, sedangkan tabel tanpa indeks 12,5739 detik | Lemah untuk menghitung, yaitu 1,3041 detik berbanding 0,2148 detik pada SQL (6,07 kali) |
| 2. Dasbor pendapatan per wilayah per jam | DuckDB langsung pada Parquet berpartisi | Hitung per wilayah dengan DuckDB 0,3748 detik berbanding SQLite 2,9043 detik (7,75 kali), dan filter satu wilayah hanya 0,0498 detik | Kurang cocok untuk membaca atau mengubah satu baris utuh, dan data yang masuk sedikit demi sedikit menimbulkan banyak file kecil |
| 3. Zona yang sering dituju dan terjangkau dalam 2 atau 3 langkah | Graph (NetworkX) dari arus antarzona | Dua langkah di graph 0,0087 detik berbanding SQL 0,0177 detik, dan k = 1 sampai 4 cukup ganti angka (74, 214, 214, 214 zona) | Berada di memori sehingga perlu dibangun ulang saat data berubah (0,5541 detik), dan tidak menyimpan atribut biaya |
| 4. Taksi dalam radius 500 meter | Indeks spasial (STRtree) | Untuk 1.000 pencarian, STRtree 0,2492 detik (ditambah 0,0745 detik membangun indeks) berbanding NumPy 0,6332 detik (1,96 kali) | Keuntungannya bergantung pada pembanding dan ukuran kotak, dan butuh koordinat padahal data TLC hanya berisi zona |
| 5. Audit keuangan (tabel tarif kembali ke tanggal 1) | Delta Lake | Bukti kualitatif dari K-7 dan Latihan 6. Versi 1 tetap terbaca utuh (150.000 baris), selisih 26,0 sama dengan 26 baris yang dikoreksi, dan restore dicatat sebagai versi 24 | File lama menumpuk (24 file di disk, 1 aktif), perlu compaction berkala, dan VACUUM dengan retention 0 jam berbahaya |

**Diagram aliran data**

```
                         +--> [Fitur 1] Key-value / tabel berindeks
                         |
trips_bersih/  ----------+--> [Fitur 2] DuckDB langsung pada Parquet --(arus antarzona)--> [Fitur 3] Graph
(Parquet berpartisi,     |
 sumber data mentah)     +--> [Fitur 4] Indeks spasial STRtree (titik jemput)
                         |
                         +--> [Fitur 5] Tabel Delta Lake (tarif, time travel)
```

**Data ganda dan cara menjaganya tetap sama**

Fitur 1, 3, 4, dan 5 sama-sama menyimpan salinan atau turunan dari `trips_bersih/`, yaitu dict untuk pencarian id, ringkasan arus antarzona, indeks titik jemput, dan tabel tarif di Delta. Hanya fitur 2 yang membaca langsung dari Parquet. Agar semua salinan tetap sama, `trips_bersih/` ditetapkan sebagai sumber data mentah yang tidak diubah, sedangkan koreksi tarif hanya ditulis ke tabel Delta. Salinan lainnya dibangun ulang dari sumber tersebut lewat satu proses batch setiap ada data baru atau versi Delta baru, sehingga tidak ada proses yang menulis langsung ke salinan. Setelah dibangun ulang, hasilnya diperiksa silang, misalnya dengan membandingkan jumlah baris dan hasil agregat seperti pemeriksaan "hasil sama?" pada K-2, K-4, dan K-5. Nomor versi Delta juga dicatat sebagai penanda salinan mana yang sudah paling baru.

## **Latihan**

### **Latihan 1**


In [ ]:
id_2000 = random.sample(range(1, len(data) + 1), 2000)
cur = con.cursor()

t0 = time.perf_counter()
_ = [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone() for i in id_2000]
t_sql = time.perf_counter() - t0

t0 = time.perf_counter()
_ = [json.loads(kv[f"perjalanan:{i}"]) for i in id_2000]
t_kv = time.perf_counter() - t0

print(f"SQL berindeks : {t_sql:.4f} s total | {t_sql / 2000 * 1e6:.2f} mikrodetik/pencarian")
print(f"Key-value     : {t_kv:.4f} s total | {t_kv / 2000 * 1e6:.2f} mikrodetik/pencarian")

SQL berindeks : 0.0357 s total | 17.83 mikrodetik/pencarian
Key-value     : 0.0116 s total | 5.79 mikrodetik/pencarian


Rata-rata waktu per pencarian untuk SQL berindeks adalah 17,83 mikrodetik, sedangkan key-value 5,79 mikrodetik, jadi key-value sekitar 3,08 kali lebih cepat. Selisih sekitar 12 mikrodetik itu tidak akan terasa oleh pengguna. Keunggulan key-value pada satu komputer sebenarnya bukan soal kecepatan, melainkan kemudahan membagi datanya ke banyak mesin.

### **Latihan 2**


In [ ]:
print(con.execute("SELECT DISTINCT metode_bayar FROM perjalanan").fetchall())  # cek nilai 'Tunai'
ZT = 236
n_doc = len(tabel.search((Q.tujuan.zona == ZT) & (Q.biaya.metode_bayar == "Tunai")))
n_sql_all = con.execute("SELECT COUNT(*) FROM perjalanan WHERE zona_tujuan=? AND metode_bayar='Tunai'", (ZT,)).fetchone()[0]
n_sql_20k = con.execute("SELECT COUNT(*) FROM perjalanan WHERE zona_tujuan=? AND metode_bayar='Tunai' AND id_perjalanan<=20000", (ZT,)).fetchone()[0]
print("TinyDB (20.001 dokumen)        :", n_doc)
print("SQL seluruh tabel (300.000)    :", n_sql_all)
print("SQL id <= 20000 (setara TinyDB):", n_sql_20k)

[('Kartu kredit',), ('Tunai',), (None,), ('Sengketa',), ('Gratis',)]
TinyDB (20.001 dokumen)        : 136
SQL seluruh tabel (300.000)    : 2098
SQL id <= 20000 (setara TinyDB): 136


TinyDB menghasilkan 136 dokumen, SQL pada seluruh tabel menghasilkan 2.098 baris, dan SQL dengan syarat id sampai 20.000 menghasilkan 136 baris. Jadi angkanya tidak sama bila dibandingkan dengan seluruh tabel, karena TinyDB hanya memuat 20.000 baris pertama sedangkan tabel SQL memuat 300.000 baris. Begitu cakupannya disamakan, hasilnya sama persis, yaitu 136.

### **Latihan 3**


In [ ]:
def median_waktu(sql, n=5):
    w = []
    for _ in range(n):
        t0 = time.perf_counter(); con_d.sql(sql).fetchall(); w.append(time.perf_counter() - t0)
    return sorted(w)[n // 2]

t_1 = median_waktu("SELECT SUM(total_bayar) FROM perjalanan")
t_4 = median_waktu("SELECT SUM(total_bayar), SUM(jarak_mil), SUM(zona_jemput), SUM(zona_tujuan) FROM perjalanan")
print(f"SUM 1 kolom  : {t_1:.4f} s")
print(f"SUM 4 kolom  : {t_4:.4f} s")
print(f"rasio 4 kolom / 1 kolom: {t_4 / t_1:.2f}")

SUM 1 kolom  : 0.1741 s
SUM 4 kolom  : 0.2955 s
rasio 4 kolom / 1 kolom: 1.70


SUM pada satu kolom memerlukan 0,1741 detik, sedangkan SUM pada empat kolom 0,2955 detik atau 1,70 kali lebih lama. Hal ini sesuai dengan column pruning, karena query hanya membaca kolom yang disebut dari 18 kolom yang ada, sehingga waktunya mengikuti jumlah kolom yang dibaca dan bukan seluruh isi baris. Waktunya naik kurang dari empat kali kemungkinan karena sebagian waktu terpakai untuk hal yang tetap, seperti membuka file.

### **Latihan 4**


In [ ]:
top_out = sorted(G.out_degree(), key=lambda kv_: (-kv_[1], kv_[0]))[:5]
print("graph (out-degree):", top_out)
top_sql4 = con_d.sql("""SELECT zona_jemput, COUNT(DISTINCT zona_tujuan) AS n_tujuan
                        FROM perjalanan GROUP BY 1 ORDER BY 2 DESC, 1 LIMIT 5""").fetchall()
print("SQL               :", top_sql4)
print("sama?", [(int(a), int(b)) for a, b in top_out] == [(int(a), int(b)) for a, b in top_sql4])

graph (out-degree): [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
SQL               : [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
sama? True


Lima zona dengan tujuan berbeda terbanyak adalah zona 132 (260 tujuan), 138 (254), 264 (241), 230 (234), dan 140 (233). Hasil dari graph dan SQL sama persis (`sama? True`), jadi keduanya menjawab pertanyaan yang sama.

### **Latihan 5**


In [ ]:
t_bangun = float(rekap.loc[rekap.langkah == "bangun indeks STRtree", "detik"].iloc[0])

def banding(sisi):
    h = sisi / 2
    kt = [box(x - h, y - h, x + h, y + h) for x, y in zip(cx, cy)]
    t0 = time.perf_counter()
    hn = []
    for x, y in zip(cx, cy):
        m = (bujur >= x - h) & (bujur <= x + h) & (lintang >= y - h) & (lintang <= y + h)
        hn.append(int(m.sum()))
    t_np = time.perf_counter() - t0
    t0 = time.perf_counter()
    hi = [len(pohon.query(k, predicate="contains")) for k in kt]
    t_idx = time.perf_counter() - t0
    print(f"sisi {sisi}: NumPy {t_np:.4f} s | STRtree {t_idx:.4f} s | NumPy/STRtree {t_np / t_idx:.2f}x "
          f"| NumPy/(STRtree+bangun) {t_np / (t_idx + t_bangun):.2f}x | hasil sama? {hn == hi} | total titik {sum(hi)}")

for sisi in (0.02, 0.01, 0.001):
    banding(sisi)

sisi 0.02: NumPy 0.5101 s | STRtree 0.6964 s | NumPy/STRtree 0.73x | NumPy/(STRtree+bangun) 0.66x | hasil sama? True | total titik 1666631
sisi 0.01: NumPy 0.5144 s | STRtree 0.1824 s | NumPy/STRtree 2.82x | NumPy/(STRtree+bangun) 2.00x | hasil sama? True | total titik 417545
sisi 0.001: NumPy 0.5070 s | STRtree 0.0117 s | NumPy/STRtree 43.31x | NumPy/(STRtree+bangun) 5.88x | hasil sama? True | total titik 4149


Perbandingan NumPy terhadap STRtree berubah cukup jauh menurut ukuran kotak. Pada sisi 0,02 derajat STRtree justru lebih lambat (0,73 kali), pada 0,01 derajat 2,82 kali lebih cepat, dan pada 0,001 derajat 43,31 kali lebih cepat. Waktu NumPy hampir tidak berubah (sekitar 0,51 detik) karena selalu memeriksa 200.000 titik, sedangkan waktu STRtree mengikuti jumlah titik yang ditemukan (1.666.631, 417.545, dan 4.149). Kotak yang lebih kecil lebih menguntungkan indeks karena makin banyak kelompok titik yang bisa dilewati, sehingga titik yang harus diperiksa jauh lebih sedikit.

### **Latihan 6**


In [ ]:
dt = DeltaTable(PATH_DELTA_LOKAL)
print("versi sebelum restore:", dt.version(), "| baris:", len(dt.to_pandas()))
dt.restore(1)
dt = DeltaTable(PATH_DELTA_LOKAL)
print("versi setelah restore:", dt.version(), "| baris:", len(dt.to_pandas()))
print([h.get("operation") for h in dt.history()][:3])

versi sebelum restore: 23 | baris: 160000
versi setelah restore: 24 | baris: 150000
['RESTORE', 'OPTIMIZE', 'WRITE']


Sebelum restore tabel berada di versi 23 dengan 160.000 baris, dan setelah restore menjadi versi 24 dengan 150.000 baris (operasi terbaru `RESTORE`). Time travel hanya membaca versi lama tanpa mengubah tabel, sedangkan restore benar-benar mengembalikan keadaan tabel ke versi 1. Restore dicatat sebagai versi baru karena log transaksi hanya bertambah, bukan dihapus. Dengan begitu riwayat tetap lengkap dan restore itu sendiri masih bisa dibatalkan.

## **Tugas Praktikum**

### **Tugas 1**
Mengumpulkan benchmark_nosql.csv dan tabel rekap K-9 yang sudah terisi, dengan
satu kalimat interpretasi untuk tiap baris.  

| Kebutuhan | Cara yang unggul | Cara yang lemah | Angka saya | Interpretasi |
|---|---|---|---|---|
| Ambil satu data lewat id | Key-value, tabel berindeks | Tabel tanpa indeks | 0,0011 detik, 0,0035 detik, 12,5739 detik | Indeks mempercepat pencarian sekitar 3.592,5 kali, sedangkan key-value hanya 3,18 kali di atas tabel berindeks. |
| Menghitung banyak baris | DuckDB pada Parquet | Key-value | DuckDB 0,3748 detik berbanding SQLite 2,9043 detik (7,75 kali), key-value 1,3041 detik berbanding SQL 0,2148 detik (6,07 kali) | Penyimpanan per kolom unggul untuk merangkum data, sedangkan key-value harus membuka setiap nilai. |
| Field opsional | Document | Tabel (NULL, migrasi) | (kualitatif) 4.090 dari 20.000 dokumen tanpa tip, 1 dokumen dengan cuaca | Document menerima bentuk data yang beragam tanpa migrasi, tetapi tidak menegur salah nama field. |
| Keterhubungan | Graph | SQL (satu JOIN per langkah) | 0,0087 detik berbanding 0,0177 detik, 74, 214, 214, 214 zona | Keduanya cepat pada 262 zona, tetapi graph unggul dalam cara bertanya. |
| Mencari lokasi | STRtree | Memeriksa semua titik | 0,2492 detik (+0,0745 detik), NumPy 0,6332 detik, loop sekitar 2.346,8 detik | Indeks 7.249,9 kali lebih cepat dari loop tetapi hanya 1,96 kali dari NumPy, jadi pembandingnya harus disebut. |
| Perubahan yang aman | Tabel Delta | Folder Parquet biasa | (kualitatif) selisih 26,0 sama dengan 26 baris, kolom asing ditolak dan versi tetap 2 | Delta punya time travel, penolakan skema, dan restore yang tidak dimiliki folder Parquet biasa. |

### **Tugas 2**


In [ ]:
# Ringkasan per zona jemput
zona_df = con_d.sql("""SELECT zona_jemput AS zona_id, MAX(wilayah_jemput) AS wilayah,
                              COUNT(*) AS jumlah_jemput, ROUND(AVG(total_bayar), 2) AS rata_bayar
                       FROM perjalanan GROUP BY zona_jemput""").df()
zona_df["wilayah"] = zona_df["wilayah"].astype(str)

# Cara 1: tabel
con_z = sqlite3.connect(":memory:")
zona_df.to_sql("zona", con_z, index=False)

# Cara 2: document (tiga tujuan teratas disatukan dalam dokumen zona)
top3 = arus.sort_values(["asal", "jumlah", "tujuan"], ascending=[True, False, True]).groupby("asal").head(3)
dok_zona = {int(r.zona_id): {"zona_id": int(r.zona_id), "wilayah": r.wilayah,
                             "jumlah_jemput": int(r.jumlah_jemput), "rata_bayar": float(r.rata_bayar),
                             "tujuan_teratas": []} for r in zona_df.itertuples(index=False)}
for r in top3.itertuples(index=False):
    if int(r.asal) in dok_zona:
        dok_zona[int(r.asal)]["tujuan_teratas"].append({"zona": int(r.tujuan), "jumlah": int(r.jumlah)})

# --- TABEL: pertanyaan mudah vs canggung
print("TABEL mudah   :", con_z.execute("SELECT * FROM zona WHERE zona_id = 236").fetchall())
def tiga_langkah_sql():
    return set(con_d.sql(f"""SELECT DISTINCT a3.tujuan FROM arus a1
        JOIN arus a2 ON a1.tujuan = a2.asal JOIN arus a3 ON a2.tujuan = a3.asal
        WHERE a1.asal = {zona_awal} AND a1.jumlah >= {MIN} AND a2.jumlah >= {MIN}
          AND a3.jumlah >= {MIN} AND a3.tujuan <> {zona_awal}""").df()["tujuan"])
t0 = time.perf_counter(); s3 = tiga_langkah_sql(); t_s3 = time.perf_counter() - t0
print(f"TABEL canggung: tepat 3 langkah (2 JOIN) -> {len(s3)} zona, {t_s3:.4f} s")

# --- DOCUMENT: mudah vs canggung
print("DOCUMENT mudah:", dok_zona[236])
from collections import Counter
t0 = time.perf_counter()
hitung = Counter(t["zona"] for d in dok_zona.values() for t in d["tujuan_teratas"])
t_dok = time.perf_counter() - t0
print(f"DOCUMENT canggung: zona paling sering jadi tujuan teratas {hitung.most_common(3)} ({t_dok:.4f} s, buka semua dokumen)")

# --- GRAPH: mudah vs canggung
def tepat_k_graph(k):
    front = {zona_awal}
    for _ in range(k):
        front = {c for b in front for c in H[b]}
    return front - {zona_awal}
t0 = time.perf_counter(); g3 = tepat_k_graph(3); t_g3 = time.perf_counter() - t0
print(f"GRAPH mudah   : tepat 3 langkah -> {len(g3)} zona, {t_g3:.4f} s | sama dengan SQL? {g3 == s3}")
print("GRAPH canggung: atribut node zona 236 =", dict(G.nodes[236]), "(rata_bayar tidak tersimpan di graph)")

TABEL mudah   : [(236, 'Manhattan', 136276, 20.64)]
TABEL canggung: tepat 3 langkah (2 JOIN) -> 214 zona, 0.0292 s
DOCUMENT mudah: {'zona_id': 236, 'wilayah': 'Manhattan', 'jumlah_jemput': 136276, 'rata_bayar': 20.64, 'tujuan_teratas': [{'zona': 237, 'jumlah': 18810}, {'zona': 236, 'jumlah': 14110}, {'zona': 161, 'jumlah': 7497}]}
DOCUMENT canggung: zona paling sering jadi tujuan teratas [(79, 20), (230, 20), (231, 18)] (0.0006 s, buka semua dokumen)
GRAPH mudah   : tepat 3 langkah -> 214 zona, 0.0007 s | sama dengan SQL? True
GRAPH canggung: atribut node zona 236 = {} (rata_bayar tidak tersimpan di graph)


Data lain yang saya pilih adalah data zona. Berikut tiga cara menyimpannya beserta satu pertanyaan yang mudah dan satu yang canggung untuk masing-masing.

| Cara | Pertanyaan yang mudah | Pertanyaan yang canggung |
|---|---|---|
| Tabel | Menampilkan data zona 236 cukup dengan satu SELECT, hasilnya (236, Manhattan, 136.276 jemput, rata-rata bayar 20,64) | Mencari zona yang terjangkau dalam tepat 3 langkah butuh 2 JOIN, hasilnya 214 zona dalam 0,0292 detik |
| Document | Membaca dokumen zona 236 utuh beserta tiga tujuan teratasnya dalam sekali baca | Mencari zona yang paling sering menjadi tujuan teratas harus membuka semua dokumen (zona 79 dan 230 masing-masing 20 kali, zona 231 sebanyak 18 kali) |
| Graph | Mencari zona yang terjangkau dalam tepat 3 langkah, hasilnya 214 zona dalam 0,0007 detik dan sama dengan hasil SQL | Membaca atribut zona 236 ternyata kosong (`{}`), jadi rata-rata bayar harus disimpan di tempat lain |

### **Tugas 3**

In [ ]:
PATH_2H = "/content/delta_dua_hari"
if os.path.exists(PATH_2H):
    shutil.rmtree(PATH_2H)

# HARI 1: tulis sebagian data
write_deltalake(PATH_2H, data_delta.iloc[:60_000], mode="overwrite")
versi_hari1 = DeltaTable(PATH_2H).version()
jumlah_h1 = len(DeltaTable(PATH_2H).to_pandas())
total_h1 = DeltaTable(PATH_2H).to_pandas()["total_bayar"].sum()
print("AKHIR HARI 1 -> versi:", versi_hari1, "| baris:", jumlah_h1)

# HARI 2: tambah data, lalu koreksi tarif Tunai di Manhattan +0.5
write_deltalake(PATH_2H, data_delta.iloc[60_000:100_000], mode="append")
dt2 = DeltaTable(PATH_2H)
n_kor = int(((data_delta.iloc[:100_000]["wilayah_jemput"] == "Manhattan") &
             (data_delta.iloc[:100_000]["metode_bayar"] == "Tunai")).sum())
dt2.update(predicate="wilayah_jemput = 'Manhattan' AND metode_bayar = 'Tunai'",
           updates={"total_bayar": "total_bayar + 0.5"})
dt2 = DeltaTable(PATH_2H)
print("AKHIR HARI 2 -> versi:", dt2.version(), "| baris:", len(dt2.to_pandas()), "| baris dikoreksi:", n_kor)

# Baca keadaan akhir hari pertama (time travel)
h1 = DeltaTable(PATH_2H, version=versi_hari1).to_pandas()
print("time travel ke akhir hari 1 -> baris:", len(h1), "| total_bayar sama dengan saat hari 1?",
      round(h1["total_bayar"].sum(), 2) == round(total_h1, 2))

# Riwayat dan isi _delta_log
for h in dt2.history():
    print(h.get("version"), h.get("operation"))
log_dir = f"{PATH_2H}/_delta_log"
print(sorted(os.listdir(log_dir)))
for f in sorted(os.listdir(log_dir)):
    print("\n==", f, "==")
    print(open(f"{log_dir}/{f}").read()[:700])

AKHIR HARI 1 -> versi: 0 | baris: 60000
AKHIR HARI 2 -> versi: 2 | baris: 100000 | baris dikoreksi: 14811
time travel ke akhir hari 1 -> baris: 60000 | total_bayar sama dengan saat hari 1? True
2 UPDATE
1 WRITE
0 WRITE
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']

== 00000000000000000000.json ==
{"commitInfo":{"timestamp":1790936136874,"operation":"WRITE","operationParameters":{"mode":"Overwrite"},"engineInfo":"delta-rs:py-1.6.6","operationMetrics":{"num_added_files":1,"num_removed_files":0,"num_partitions":0,"num_added_rows":60000,"execution_time_ms":20,"num_retries":0},"clientVersion":"delta-rs.py-1.6.6"}}
{"protocol":{"minReaderVersion":3,"minWriterVersion":7,"readerFeatures":["timestampNtz"],"writerFeatures":["timestampNtz"]}}
{"metaData":{"id":"e113fd7c-ba13-4b0d-935c-317b57036c3c","name":null,"description":null,"format":{"provider":"parquet","options":{}},"schemaString":"{\"type\":\"struct\",\"fields\":[{\"name\":\"waktu_jemput\",\"type\

Pada simulasi dua hari ini, hari pertama menulis 60.000 baris sehingga tabel berada di versi 0. Pada hari kedua saya menambah 40.000 baris (versi 1), lalu mengoreksi 14.811 baris perjalanan Tunai di Manhattan dengan menambah 0,5 pada `total_bayar`, sehingga tabel berada di versi 2 dengan 100.000 baris. Time travel ke versi akhir hari pertama menghasilkan 60.000 baris dengan total `total_bayar` yang sama seperti saat hari pertama (`True`), jadi keadaan tabel di akhir hari pertama tetap bisa dibaca meskipun koreksi sudah dilakukan.

Catatan, pada output isi tiap berkas di `_delta_log/` hanya ditampilkan 700 karakter pertama, sehingga skema tabel dan statistik file tidak tampil seluruhnya. Ringkasan ketiga berkas diambil dari bagian `commitInfo` yang terlihat pada output.

| Berkas | Operasi | Isi penting |
|---|---|---|
| 00000000000000000000.json | WRITE (overwrite) | Menulis data awal hari pertama, 1 file ditambahkan dengan 60.000 baris |
| 00000000000000000001.json | WRITE (append) | Menambah data hari kedua, 1 file ditambahkan dengan 40.000 baris |
| 00000000000000000002.json | UPDATE | 14.811 baris diubah dan 85.189 baris disalin, 2 file dikeluarkan dan 1 file baru ditambahkan (100.000 baris) |

### **Tugas 4**


In [ ]:
shutil.rmtree(PATH_DELTA_DRIVE, ignore_errors=True)
shutil.copytree(PATH_DELTA_LOKAL, PATH_DELTA_DRIVE)
print(shutil.make_archive(f"{DIR_SIMPAN}/trips_delta", "zip", PATH_DELTA_DRIVE))

/content/drive/MyDrive/BigData/Praktikum4/trips_delta.zip


**Pengumpulan arsip trips_delta.zip**

Dijalankan paling akhir karena salinan tabel Delta di Google Drive masih berhenti di versi 2 (hasil K-7), jadi folder lamanya dihapus lalu tabel lokal yang sudah mencapai keadaan akhir disalin ulang. Setelah itu folder tersebut diarsipkan menjadi `trips_delta.zip` di folder Praktikum 4 untuk dikumpulkan bersama notebook, `benchmark_nosql.csv`, dan laporan PDF.